## Convert Raw Training Images to Ordered PNG Files

This script converts a folder of raw training images into consistently named `.png` files for MATLAB labeling and later COCO annotation conversion.

It accepts common image formats such as `.png`, `.jpg`, `.jpeg`, `.tif`, `.tiff`, and `.bmp`, sorts the files in natural filename order, converts each image to PNG, and renames the outputs using a numbered sequence such as `bubble_000001.png`, `bubble_000002.png`, and so on.

The script also creates a `filename_map.csv` file that records the original filename and corresponding new PNG filename. This makes it easier to trace converted training images back to their original source files.

Example output:

```text
bubble_000001.png
bubble_000002.png
bubble_000003.png
filename_map.csv
```

Use this script before labeling images in MATLAB so that the exported pixel label masks can be matched reliably during COCO conversion.


In [ ]:
# ============================================================
# Convert training images to ordered PNG files
#
# Purpose:
#   Prepares a folder of raw training images for MATLAB labeling
#   and later COCO conversion.
#
# Example:
#   python training/convert_images_to_png.py ^
#     --input-dir "C:\path\to\raw_images" ^
#     --output-dir "C:\path\to\png_training_images" ^
#     --prefix bubble
#
# Output:
#   bubble_000001.png
#   bubble_000002.png
#   ...
#   filename_map.csv
# ============================================================

import argparse
import csv
from pathlib import Path

from PIL import Image, ImageOps


SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp",
}


def natural_key(path):
    """
    Sort filenames in human order:
        image2.tif before image10.tif
    """
    import re

    text = path.name
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", text)
    ]


def find_image_files(input_dir):
    """
    Finds supported image files in the input folder.
    """
    files = [
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    return sorted(files, key=natural_key)


def convert_to_png(input_path, output_path):
    """
    Converts one image to PNG.

    ImageOps.exif_transpose handles cases where jpg files have orientation
    metadata instead of actual rotated pixels.
    """
    with Image.open(input_path) as img:
        img = ImageOps.exif_transpose(img)

        # Convert unsupported modes into normal 8-bit RGB or grayscale.
        if img.mode in ("RGBA", "LA"):
            # Keep alpha if present.
            converted = img
        elif img.mode in ("L", "I;16", "I"):
            converted = img.convert("L")
        else:
            converted = img.convert("RGB")

        converted.save(output_path)


def main():
    parser = argparse.ArgumentParser(
        description="Convert a folder of training images to ordered PNG files."
    )

    parser.add_argument(
        "--input-dir",
        required=True,
        help="Folder containing raw training images."
    )

    parser.add_argument(
        "--output-dir",
        required=True,
        help="Folder where ordered PNG files will be saved."
    )

    parser.add_argument(
        "--prefix",
        default="bubble",
        help="Output filename prefix. Default: bubble"
    )

    parser.add_argument(
        "--start-index",
        type=int,
        default=1,
        help="Starting index for output filenames. Default: 1"
    )

    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Allow overwriting existing PNG files in the output folder."
    )

    args = parser.parse_args()

    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder does not exist: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)

    image_files = find_image_files(input_dir)

    if len(image_files) == 0:
        raise RuntimeError(f"No supported image files found in: {input_dir}")

    print(f"Found {len(image_files)} image files.")

    map_rows = []

    for count, input_path in enumerate(image_files, start=args.start_index):
        output_name = f"{args.prefix}_{count:06d}.png"
        output_path = output_dir / output_name

        if output_path.exists() and not args.overwrite:
            raise FileExistsError(
                f"Output file already exists: {output_path}\n"
                "Use --overwrite if you want to replace existing files."
            )

        convert_to_png(input_path, output_path)

        map_rows.append({
            "new_filename": output_name,
            "original_filename": input_path.name,
            "original_path": str(input_path),
        })

        print(f"{input_path.name} -> {output_name}")

    map_path = output_dir / "filename_map.csv"

    with open(map_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["new_filename", "original_filename", "original_path"]
        )
        writer.writeheader()
        writer.writerows(map_rows)

    print("\nDone.")
    print(f"Saved PNG images to: {output_dir}")
    print(f"Saved filename map to: {map_path}")


if __name__ == "__main__":
    main()